In [16]:
import math
import numpy as np
mnist_file = "mnist_test.txt"
File_data = np.loadtxt(mnist_file, dtype=int)

number_of_examples = 100
# Remove the class cols 
File_data = File_data[:, :-1]
AXI_bus_size = 64


print("Number of features:\t\t", File_data[0].shape[0])
print("Packets:\t\t\t", File_data[0].shape[0]/AXI_bus_size)
print("Number of Extra packet(s):\t", math.ceil(File_data[0].shape[0]/AXI_bus_size) - math.floor(File_data[0].shape[0]/AXI_bus_size))
print("-------------------------------------")
print("Number of packets required:\t", math.ceil(File_data[0].shape[0]/AXI_bus_size))
print("-------------------------------------")

# Put the data into 32 int packets 

mnist_packets = []
packet_32 = []
packet_counter = 0 

for j in range(number_of_examples):
	for i in range(File_data[j].shape[0]):
		if packet_counter <= AXI_bus_size-1: 
			packet_32.append(File_data[j][i])
		else: 
			mnist_packets.append(packet_32)
			packet_32 = []
			# print(i)
			packet_counter = 0
			packet_32.append(File_data[j][i])

		packet_counter += 1

	# If datapoint is complete and packet is not fully filled...
	# Fill the remainder with zeros 
	if(len(packet_32) != 0):
		# print("Half filled packet: ", len(packet_32))
		remainder_zero_fill = AXI_bus_size -len(packet_32)   
		for l in range(remainder_zero_fill):
			packet_32.append(0)
		
		mnist_packets.append(packet_32)
		packet_counter = 0
		packet_32 = []

	# for l in range((math.ceil(File_data[0].shape[0]/32) - math.floor(File_data[0].shape[0]/32))):
	# 	mnist_packets.append(zero_list) 

# convert the list of lists to a numpy array 
mnist_packets_np_1 = np.array(mnist_packets)

# mnist_packets_np  = np.fliplr(mnist_packets_np_1)
# mnist_packets_np = np.insert(mnist_packets_np, 0, number_of_examples)
# print(mnist_packets_np)
# now convert each 32 number array to its equivalent binary 
# this will be the 32 bit packet sent each time 

mnist_packets_np_bits =np.packbits(mnist_packets_np_1, axis=-1,  bitorder='little')
mnist_packets_np_bits.dtype = np.uint64
# mnist_packets_np_bits = np.insert(mnist_packets_np_bits, 0, number_of_examples)
# print(mnist_packets_np_bits)
# print(mnist_packets_np_bits.shape)

Number of features:		 784
Packets:			 12.25
Number of Extra packet(s):	 1
-------------------------------------
Number of packets required:	 13
-------------------------------------


In [15]:
from pynq import Overlay
# from pynq import Xlnk
from pynq.lib import dma
from pynq import allocate
import numpy as np
import pynq

ol = Overlay("CoTM_inference_System_V1.bit")

In [17]:
ol.ip_dict

{'axi_dma_0': {'type': 'xilinx.com:ip:axi_dma:7.1',
  'mem_id': 'S_AXI_LITE',
  'memtype': 'REGISTER',
  'gpio': {},
  'interrupts': {},
  'parameters': {'C_DLYTMR_RESOLUTION': '125',
   'C_ENABLE_MULTI_CHANNEL': '0',
   'C_FAMILY': 'zynq',
   'C_INCLUDE_MM2S': '1',
   'C_INCLUDE_MM2S_DRE': '0',
   'C_INCLUDE_MM2S_SF': '1',
   'C_INCLUDE_S2MM': '1',
   'C_INCLUDE_S2MM_DRE': '0',
   'C_INCLUDE_S2MM_SF': '1',
   'C_INCLUDE_SG': '0',
   'C_INCREASE_THROUGHPUT': '0',
   'C_MICRO_DMA': '0',
   'C_MM2S_BURST_SIZE': '16',
   'C_M_AXIS_MM2S_CNTRL_TDATA_WIDTH': '32',
   'C_M_AXIS_MM2S_TDATA_WIDTH': '64',
   'C_M_AXI_MM2S_ADDR_WIDTH': '64',
   'C_M_AXI_MM2S_DATA_WIDTH': '64',
   'C_M_AXI_S2MM_ADDR_WIDTH': '64',
   'C_M_AXI_S2MM_DATA_WIDTH': '64',
   'C_M_AXI_SG_ADDR_WIDTH': '64',
   'C_M_AXI_SG_DATA_WIDTH': '32',
   'C_NUM_MM2S_CHANNELS': '1',
   'C_NUM_S2MM_CHANNELS': '1',
   'C_PRMRY_IS_ACLK_ASYNC': '0',
   'C_S2MM_BURST_SIZE': '16',
   'C_SG_INCLUDE_STSCNTRL_STRM': '0',
   'C_SG_LENGTH_WIDTH'

In [36]:
ol.axi_dma_0?

In [18]:
dma = ol.axi_dma_0
dma_send = ol.axi_dma_0.sendchannel
dma_recv = ol.axi_dma_0.recvchannel

In [19]:
import time
image_num = 100

print("-------------------------------------")
# print(dma.register_map)

input_buffer = allocate(shape=(mnist_packets_np_bits.shape[0],), dtype=np.uint64)
for i in range(mnist_packets_np_bits.shape[0]):
    input_buffer[i] = mnist_packets_np_bits[i] 
#     print(input_buffer[i])

print("-------------------------------------")
# # start_time = time.time()

# # time.sleep(5)
dma_send.transfer(input_buffer)
# # send_time = time.time()

output_buffer = allocate(shape=(image_num,), dtype=np.uint64)
dma_recv.transfer(output_buffer)
# time.sleep(5)
# dma.sendchannel.wait()
# dma.recvchannel.wait()

# stop_time = time.time()

# hw_exec_time = stop_time-start_time
# send_exec_time = send_time-start_time
# ip_exec_time = stop_time-send_time

# print("-------------------------------------")
# print(dma.register_map)


print("-------------------------------------")
for i in range(image_num):
    print(i,':',output_buffer[i])

# print("-------------------------------------")
# print('Send execution time: ',send_exec_time)
# print('Hardware execution time: ',hw_exec_time)
# print('IP core execution time: ',ip_exec_time)

-------------------------------------
-------------------------------------
-------------------------------------
0 : 0
1 : 9
2 : 2
3 : 4
4 : 2
5 : 9
6 : 9
7 : 6
8 : 6
9 : 0
10 : 5
11 : 7
12 : 4
13 : 1
14 : 0
15 : 1
16 : 4
17 : 5
18 : 0
19 : 0
20 : 1
21 : 9
22 : 7
23 : 4
24 : 9
25 : 6
26 : 4
27 : 5
28 : 4
29 : 0
30 : 7
31 : 4
32 : 0
33 : 1
34 : 3
35 : 1
36 : 3
37 : 4
38 : 7
39 : 2
40 : 7
41 : 1
42 : 3
43 : 1
44 : 1
45 : 7
46 : 4
47 : 2
48 : 3
49 : 5
50 : 1
51 : 2
52 : 4
53 : 4
54 : 6
55 : 3
56 : 5
57 : 5
58 : 6
59 : 0
60 : 4
61 : 1
62 : 9
63 : 5
64 : 7
65 : 9
66 : 3
67 : 7
68 : 9
69 : 3
70 : 4
71 : 3
72 : 0
73 : 7
74 : 0
75 : 2
76 : 9
77 : 1
78 : 7
79 : 3
80 : 2
81 : 7
82 : 9
83 : 6
84 : 2
85 : 7
86 : 8
87 : 4
88 : 7
89 : 5
90 : 6
91 : 1
92 : 3
93 : 6
94 : 8
95 : 3
96 : 1
97 : 4
98 : 1
99 : 7


In [63]:
import time
image_num = 9999

print("-------------------------------------")
# print(dma.register_map)
start_time = time.time()
input_buffer = allocate(shape=(mnist_packets_np_bits.shape[0],), dtype=np.uint64)
for i in range(mnist_packets_np_bits.shape[0]):
    input_buffer[i] = mnist_packets_np_bits[i] 
stop_time = time.time()


-------------------------------------


In [64]:
exec_time = stop_time-start_time
print('buffer allocate time: ',exec_time)

buffer allocate time:  0.9020655155181885


In [65]:
start_time = time.time()
dma_send.transfer(input_buffer)
# # send_time = time.time()

output_buffer = allocate(shape=(image_num,), dtype=np.uint64)
dma_recv.transfer(output_buffer)
stop_time = time.time()

In [66]:
exec_time = stop_time-start_time
print('buffer allocate time: ',exec_time)

buffer allocate time:  0.00830221176147461


In [7]:
# time.sleep(5)
dma_send.transfer(input_buffer)
output_buffer = allocate(shape=(image_num,), dtype=np.uint64)
dma_recv.transfer(output_buffer)
# time.sleep(10)

# time.sleep(5)
# dma_send.transfer(input_buffer)
# output_buffer = allocate(shape=(image_num,), dtype=np.uint64)
# dma_recv.transfer(output_buffer)
# time.sleep(5)

RuntimeError: DMA channel not halted

In [67]:
for i in range(image_num):
    print(i,':',output_buffer[i])

0 : 0
1 : 7
2 : 6
3 : 6
4 : 3
5 : 1
6 : 5
7 : 7
8 : 1
9 : 7
10 : 5
11 : 5
12 : 5
13 : 4
14 : 5
15 : 0
16 : 6
17 : 2
18 : 1
19 : 5
20 : 5
21 : 7
22 : 7
23 : 7
24 : 7
25 : 6
26 : 9
27 : 7
28 : 6
29 : 5
30 : 1
31 : 1
32 : 5
33 : 1
34 : 7
35 : 5
36 : 2
37 : 1
38 : 6
39 : 3
40 : 7
41 : 0
42 : 7
43 : 2
44 : 1
45 : 3
46 : 5
47 : 1
48 : 1
49 : 7
50 : 1
51 : 5
52 : 3
53 : 1
54 : 1
55 : 5
56 : 2
57 : 1
58 : 2
59 : 5
60 : 1
61 : 3
62 : 3
63 : 0
64 : 2
65 : 4
66 : 5
67 : 0
68 : 6
69 : 2
70 : 1
71 : 0
72 : 5
73 : 1
74 : 5
75 : 7
76 : 7
77 : 6
78 : 0
79 : 5
80 : 6
81 : 6
82 : 7
83 : 1
84 : 5
85 : 2
86 : 1
87 : 5
88 : 2
89 : 7
90 : 6
91 : 2
92 : 1
93 : 5
94 : 6
95 : 2
96 : 7
97 : 5
98 : 1
99 : 1
100 : 3
101 : 3
102 : 3
103 : 6
104 : 1
105 : 1
106 : 5
107 : 6
108 : 7
109 : 5
110 : 2
111 : 5
112 : 7
113 : 3
114 : 7
115 : 1
116 : 6
117 : 2
118 : 1
119 : 5
120 : 6
121 : 7
122 : 2
123 : 5
124 : 1
125 : 1
126 : 5
127 : 5
128 : 1
129 : 6
130 : 9
131 : 7
132 : 1
133 : 5
134 : 2
135 : 1
136 : 2
137 : 1
138 : 

1282 : 6
1283 : 1
1284 : 1
1285 : 1
1286 : 5
1287 : 2
1288 : 7
1289 : 1
1290 : 6
1291 : 1
1292 : 1
1293 : 1
1294 : 1
1295 : 1
1296 : 5
1297 : 9
1298 : 7
1299 : 6
1300 : 6
1301 : 7
1302 : 1
1303 : 3
1304 : 6
1305 : 1
1306 : 0
1307 : 5
1308 : 1
1309 : 1
1310 : 7
1311 : 7
1312 : 6
1313 : 1
1314 : 1
1315 : 1
1316 : 6
1317 : 2
1318 : 1
1319 : 5
1320 : 1
1321 : 1
1322 : 5
1323 : 5
1324 : 3
1325 : 6
1326 : 0
1327 : 2
1328 : 1
1329 : 6
1330 : 7
1331 : 5
1332 : 5
1333 : 1
1334 : 1
1335 : 5
1336 : 3
1337 : 1
1338 : 6
1339 : 5
1340 : 1
1341 : 1
1342 : 5
1343 : 7
1344 : 1
1345 : 5
1346 : 1
1347 : 7
1348 : 3
1349 : 7
1350 : 7
1351 : 5
1352 : 0
1353 : 7
1354 : 0
1355 : 6
1356 : 5
1357 : 7
1358 : 5
1359 : 1
1360 : 1
1361 : 1
1362 : 2
1363 : 1
1364 : 6
1365 : 8
1366 : 7
1367 : 0
1368 : 5
1369 : 6
1370 : 1
1371 : 2
1372 : 7
1373 : 1
1374 : 1
1375 : 3
1376 : 7
1377 : 0
1378 : 0
1379 : 3
1380 : 6
1381 : 1
1382 : 1
1383 : 2
1384 : 5
1385 : 2
1386 : 5
1387 : 5
1388 : 7
1389 : 7
1390 : 6
1391 : 2
1392 : 7
1

2651 : 6
2652 : 9
2653 : 7
2654 : 6
2655 : 6
2656 : 2
2657 : 5
2658 : 5
2659 : 1
2660 : 7
2661 : 7
2662 : 2
2663 : 5
2664 : 6
2665 : 2
2666 : 7
2667 : 6
2668 : 6
2669 : 5
2670 : 1
2671 : 5
2672 : 5
2673 : 1
2674 : 5
2675 : 7
2676 : 7
2677 : 6
2678 : 6
2679 : 7
2680 : 1
2681 : 6
2682 : 2
2683 : 1
2684 : 5
2685 : 1
2686 : 6
2687 : 3
2688 : 7
2689 : 1
2690 : 6
2691 : 6
2692 : 7
2693 : 6
2694 : 5
2695 : 2
2696 : 1
2697 : 5
2698 : 1
2699 : 1
2700 : 5
2701 : 4
2702 : 1
2703 : 6
2704 : 2
2705 : 1
2706 : 6
2707 : 6
2708 : 2
2709 : 2
2710 : 5
2711 : 5
2712 : 1
2713 : 5
2714 : 7
2715 : 1
2716 : 6
2717 : 7
2718 : 1
2719 : 1
2720 : 1
2721 : 2
2722 : 7
2723 : 5
2724 : 1
2725 : 7
2726 : 7
2727 : 5
2728 : 7
2729 : 6
2730 : 1
2731 : 7
2732 : 0
2733 : 6
2734 : 2
2735 : 2
2736 : 5
2737 : 7
2738 : 1
2739 : 7
2740 : 5
2741 : 1
2742 : 5
2743 : 0
2744 : 7
2745 : 1
2746 : 2
2747 : 2
2748 : 1
2749 : 5
2750 : 1
2751 : 7
2752 : 0
2753 : 5
2754 : 1
2755 : 6
2756 : 8
2757 : 7
2758 : 1
2759 : 6
2760 : 2
2761 : 1
2

4010 : 5
4011 : 5
4012 : 1
4013 : 5
4014 : 1
4015 : 1
4016 : 6
4017 : 8
4018 : 7
4019 : 0
4020 : 7
4021 : 7
4022 : 1
4023 : 5
4024 : 1
4025 : 1
4026 : 5
4027 : 2
4028 : 1
4029 : 0
4030 : 7
4031 : 7
4032 : 1
4033 : 6
4034 : 2
4035 : 1
4036 : 1
4037 : 1
4038 : 1
4039 : 9
4040 : 7
4041 : 6
4042 : 0
4043 : 2
4044 : 3
4045 : 7
4046 : 5
4047 : 6
4048 : 1
4049 : 5
4050 : 7
4051 : 1
4052 : 7
4053 : 5
4054 : 1
4055 : 0
4056 : 7
4057 : 1
4058 : 7
4059 : 6
4060 : 7
4061 : 1
4062 : 5
4063 : 5
4064 : 6
4065 : 5
4066 : 7
4067 : 2
4068 : 6
4069 : 9
4070 : 7
4071 : 6
4072 : 5
4073 : 6
4074 : 1
4075 : 7
4076 : 1
4077 : 1
4078 : 3
4079 : 5
4080 : 1
4081 : 6
4082 : 2
4083 : 7
4084 : 6
4085 : 6
4086 : 7
4087 : 1
4088 : 2
4089 : 5
4090 : 1
4091 : 7
4092 : 3
4093 : 5
4094 : 2
4095 : 1
4096 : 7
4097 : 1
4098 : 6
4099 : 6
4100 : 1
4101 : 5
4102 : 5
4103 : 1
4104 : 5
4105 : 3
4106 : 1
4107 : 2
4108 : 9
4109 : 7
4110 : 1
4111 : 6
4112 : 2
4113 : 1
4114 : 2
4115 : 7
4116 : 7
4117 : 5
4118 : 3
4119 : 7
4120 : 6
4

5384 : 7
5385 : 6
5386 : 6
5387 : 5
5388 : 5
5389 : 2
5390 : 1
5391 : 1
5392 : 3
5393 : 1
5394 : 6
5395 : 5
5396 : 7
5397 : 0
5398 : 6
5399 : 3
5400 : 1
5401 : 1
5402 : 1
5403 : 6
5404 : 1
5405 : 7
5406 : 1
5407 : 1
5408 : 7
5409 : 1
5410 : 5
5411 : 6
5412 : 7
5413 : 5
5414 : 5
5415 : 5
5416 : 7
5417 : 4
5418 : 2
5419 : 3
5420 : 6
5421 : 8
5422 : 2
5423 : 6
5424 : 4
5425 : 7
5426 : 1
5427 : 5
5428 : 7
5429 : 1
5430 : 1
5431 : 3
5432 : 1
5433 : 6
5434 : 9
5435 : 7
5436 : 6
5437 : 6
5438 : 7
5439 : 1
5440 : 2
5441 : 5
5442 : 7
5443 : 1
5444 : 9
5445 : 1
5446 : 5
5447 : 4
5448 : 1
5449 : 6
5450 : 6
5451 : 5
5452 : 1
5453 : 2
5454 : 5
5455 : 1
5456 : 7
5457 : 1
5458 : 7
5459 : 6
5460 : 4
5461 : 7
5462 : 7
5463 : 2
5464 : 1
5465 : 1
5466 : 3
5467 : 1
5468 : 1
5469 : 1
5470 : 1
5471 : 3
5472 : 5
5473 : 0
5474 : 1
5475 : 5
5476 : 6
5477 : 1
5478 : 5
5479 : 6
5480 : 5
5481 : 1
5482 : 7
5483 : 7
5484 : 5
5485 : 6
5486 : 4
5487 : 7
5488 : 5
5489 : 6
5490 : 7
5491 : 1
5492 : 5
5493 : 2
5494 : 1
5

6755 : 1
6756 : 5
6757 : 9
6758 : 3
6759 : 6
6760 : 1
6761 : 1
6762 : 0
6763 : 2
6764 : 2
6765 : 5
6766 : 2
6767 : 5
6768 : 1
6769 : 5
6770 : 9
6771 : 3
6772 : 6
6773 : 4
6774 : 7
6775 : 6
6776 : 6
6777 : 7
6778 : 1
6779 : 1
6780 : 7
6781 : 1
6782 : 4
6783 : 9
6784 : 3
6785 : 6
6786 : 7
6787 : 2
6788 : 6
6789 : 6
6790 : 2
6791 : 5
6792 : 5
6793 : 7
6794 : 5
6795 : 5
6796 : 3
6797 : 7
6798 : 6
6799 : 8
6800 : 5
6801 : 5
6802 : 6
6803 : 7
6804 : 1
6805 : 5
6806 : 2
6807 : 5
6808 : 1
6809 : 7
6810 : 7
6811 : 6
6812 : 0
6813 : 7
6814 : 6
6815 : 6
6816 : 7
6817 : 5
6818 : 5
6819 : 5
6820 : 1
6821 : 5
6822 : 3
6823 : 1
6824 : 6
6825 : 3
6826 : 7
6827 : 6
6828 : 6
6829 : 6
6830 : 2
6831 : 2
6832 : 5
6833 : 1
6834 : 1
6835 : 1
6836 : 3
6837 : 6
6838 : 2
6839 : 7
6840 : 6
6841 : 6
6842 : 7
6843 : 1
6844 : 5
6845 : 7
6846 : 1
6847 : 5
6848 : 1
6849 : 7
6850 : 6
6851 : 4
6852 : 7
6853 : 6
6854 : 2
6855 : 7
6856 : 1
6857 : 1
6858 : 1
6859 : 1
6860 : 7
6861 : 7
6862 : 7
6863 : 6
6864 : 7
6865 : 1
6

8133 : 1
8134 : 7
8135 : 9
8136 : 1
8137 : 6
8138 : 6
8139 : 1
8140 : 6
8141 : 2
8142 : 7
8143 : 1
8144 : 2
8145 : 5
8146 : 1
8147 : 1
8148 : 5
8149 : 7
8150 : 1
8151 : 0
8152 : 7
8153 : 0
8154 : 6
8155 : 7
8156 : 1
8157 : 2
8158 : 5
8159 : 1
8160 : 9
8161 : 7
8162 : 1
8163 : 1
8164 : 0
8165 : 7
8166 : 1
8167 : 6
8168 : 7
8169 : 1
8170 : 1
8171 : 5
8172 : 7
8173 : 5
8174 : 9
8175 : 7
8176 : 6
8177 : 6
8178 : 1
8179 : 1
8180 : 6
8181 : 5
8182 : 1
8183 : 5
8184 : 3
8185 : 1
8186 : 7
8187 : 7
8188 : 7
8189 : 6
8190 : 4
8191 : 1
8192 : 6
8193 : 1
8194 : 7
8195 : 5
8196 : 2
8197 : 5
8198 : 1
8199 : 1
8200 : 3
8201 : 1
8202 : 5
8203 : 2
8204 : 7
8205 : 1
8206 : 6
8207 : 7
8208 : 1
8209 : 1
8210 : 5
8211 : 7
8212 : 1
8213 : 1
8214 : 1
8215 : 0
8216 : 7
8217 : 7
8218 : 6
8219 : 3
8220 : 7
8221 : 1
8222 : 1
8223 : 2
8224 : 1
8225 : 5
8226 : 4
8227 : 1
8228 : 6
8229 : 6
8230 : 7
8231 : 6
8232 : 6
8233 : 5
8234 : 1
8235 : 2
8236 : 5
8237 : 4
8238 : 7
8239 : 1
8240 : 5
8241 : 2
8242 : 1
8243 : 5
8

9502 : 6
9503 : 4
9504 : 7
9505 : 6
9506 : 1
9507 : 2
9508 : 1
9509 : 2
9510 : 5
9511 : 6
9512 : 1
9513 : 7
9514 : 3
9515 : 6
9516 : 3
9517 : 7
9518 : 6
9519 : 6
9520 : 7
9521 : 5
9522 : 5
9523 : 7
9524 : 1
9525 : 7
9526 : 1
9527 : 7
9528 : 6
9529 : 4
9530 : 1
9531 : 1
9532 : 5
9533 : 7
9534 : 5
9535 : 5
9536 : 7
9537 : 1
9538 : 1
9539 : 3
9540 : 1
9541 : 2
9542 : 1
9543 : 1
9544 : 1
9545 : 1
9546 : 2
9547 : 1
9548 : 3
9549 : 7
9550 : 1
9551 : 1
9552 : 4
9553 : 1
9554 : 6
9555 : 3
9556 : 7
9557 : 5
9558 : 0
9559 : 2
9560 : 7
9561 : 5
9562 : 1
9563 : 1
9564 : 5
9565 : 9
9566 : 3
9567 : 6
9568 : 7
9569 : 7
9570 : 5
9571 : 1
9572 : 2
9573 : 2
9574 : 5
9575 : 1
9576 : 1
9577 : 6
9578 : 4
9579 : 4
9580 : 6
9581 : 0
9582 : 7
9583 : 7
9584 : 5
9585 : 7
9586 : 1
9587 : 2
9588 : 5
9589 : 1
9590 : 5
9591 : 4
9592 : 3
9593 : 6
9594 : 8
9595 : 7
9596 : 6
9597 : 6
9598 : 1
9599 : 1
9600 : 2
9601 : 5
9602 : 1
9603 : 7
9604 : 9
9605 : 7
9606 : 2
9607 : 6
9608 : 1
9609 : 6
9610 : 6
9611 : 7
9612 : 1
9

In [17]:
# from pynq import allocate
# import numpy as np
import time
image_num = 9999
run_round = 1000
print("-------------------------------------")
print(dma.register_map)

send_exec_time_sum = 0
hw_exec_time_sum = 0
ip_exec_time_sum = 0

for rounds in range(run_round):
    input_buffer = allocate(shape=(mnist_packets_np_bits.shape[0],), dtype=np.uint64)
    for i in range(mnist_packets_np_bits.shape[0]):
        input_buffer[i] = mnist_packets_np_bits[i] 
    #     print(input_buffer[i])

#     print("-------------------------------------")
    start_time = time.time()
    dma_send.transfer(input_buffer)
    send_time = time.time()

    output_buffer = allocate(shape=(image_num,), dtype=np.uint64)
    dma_recv.transfer(output_buffer)

    stop_time = time.time()

    hw_exec_time = stop_time-start_time
    send_exec_time = send_time-start_time
    ip_exec_time = stop_time-send_time

#     print("-------------------------------------")
#     print(dma.register_map)

#     print("-------------------------------------")
#     for i in range(image_num):
    #     print(format(image_num, '06x')+'x'+format(output_buffer[i], '01x'))
#         print(i,':',output_buffer[i])
    
#     send_exec_time_sum = send_exec_time + send_exec_time
    hw_exec_time_sum = hw_exec_time_sum + hw_exec_time
#     ip_exec_time_sum = ip_exec_time + ip_exec_time
    
    print("-------------------------------------")
    print('Send execution time: ',send_exec_time)
    print('Hardware execution time: ',hw_exec_time)
    print('IP core execution time: ',ip_exec_time)
    del input_buffer, output_buffer

print("-------------------------------------")
# print('Average send execution time: ',send_exec_time_sum/run_round)
print('Average hardware execution time: ',hw_exec_time_sum/run_round)
# print('Average IP core execution time: ',ip_exec_time_sum/run_round)

-------------------------------------
RegisterMap {
  MM2S_DMACR = Register(RS=1, Reset=0, Keyhole=0, Cyclic_BD_Enable=0, IOC_IrqEn=0, Dly_IrqEn=0, Err_IrqEn=0, IRQThreshold=1, IRQDelay=0),
  MM2S_DMASR = Register(Halted=0, Idle=0, SGIncld=0, DMAIntErr=0, DMASlvErr=0, DMADecErr=0, SGIntErr=0, SGSlvErr=0, SGDecErr=0, IOC_Irq=0, Dly_Irq=0, Err_Irq=0, IRQThresholdSts=0, IRQDelaySts=0),
  MM2S_CURDESC = Register(Current_Descriptor_Pointer=0),
  MM2S_CURDESC_MSB = Register(Current_Descriptor_Pointer=0),
  MM2S_TAILDESC = Register(Tail_Descriptor_Pointer=0),
  MM2S_TAILDESC_MSB = Register(Tail_Descriptor_Pointer=0),
  MM2S_SA = Register(Source_Address=0),
  MM2S_SA_MSB = Register(Source_Address=0),
  MM2S_LENGTH = Register(Length=0),
  SG_CTL = Register(SG_CACHE=0, SG_USER=0),
  S2MM_DMACR = Register(RS=1, Reset=0, Keyhole=0, Cyclic_BD_Enable=0, IOC_IrqEn=0, Dly_IrqEn=0, Err_IrqEn=0, IRQThreshold=1, IRQDelay=0),
  S2MM_DMASR = Register(Halted=0, Idle=0, SGIncld=0, DMAIntErr=0, DMASlvErr=0, D

KeyboardInterrupt: 

In [44]:
del input_buffer, output_buffer

In [45]:
dma_send.idle

True

In [46]:
dma_recv.idle

False

In [47]:
dma.sendchannel.wait()
dma.recvchannel.wait()

KeyboardInterrupt: 